# 🧠 Алгоритми та структури даних · Урок 6 — Парадигми проєктування алгоритмів

Це «мислення» алгоритміста: не окремі алгоритми, а **підходи**, якими розв'язують нові задачі —
«розділяй і володарюй», жадібні, **динамічне програмування** та **backtracking**. Наприкінці —
короткий огляд рядкових алгоритмів, теорії чисел і роботи з бітами.

**Передумова:** [Урок 1 (рекурсія, складність)](lektsiya-1-skladnist-ta-rekursiya.ipynb).

### Що ви вмітимете
- впізнавати й застосовувати **4 парадигми**;
- писати **DP** (мемоізація й табуляція) для класичних задач;
- розв'язувати задачі **перебором з відсіканням** (backtracking).

## 1. Розділяй і володарюй (divide & conquer)

Три кроки: **поділити** задачу на менші підзадачі → **розв'язати** їх (рекурсивно) → **об'єднати**
результати. Так працюють merge sort, quick sort, двійковий пошук.

Приклад — **швидке піднесення до степеня** `x^n` за **O(log n)** замість наївних O(n) множень:
`x^n = (x^(n/2))²`.

In [1]:
def fast_power(x, n):
    if n == 0:
        return 1
    half = fast_power(x, n // 2)      # розв'язуємо half-задачу ОДИН раз
    if n % 2 == 0:
        return half * half            # об'єднуємо: (x^(n/2))^2
    return half * half * x            # непарний степінь -> ще один множник

print("2^10 =", fast_power(2, 10))    # 1024
print("3^13 =", fast_power(3, 13))    # 1594323
print("перевірка:", 3 ** 13)

2^10 = 1024
3^13 = 1594323
перевірка: 1594323


## 2. Жадібні алгоритми (greedy)

**Жадібний** алгоритм на кожному кроці робить **локально найкращий** вибір, сподіваючись на
глобальний оптимум. Простий і швидкий — **але не завжди правильний**: працює лише для задач з
особливою структурою (де локальний оптимум гарантує глобальний).

Класика, де жадібність **правильна**: вибір максимуму подій без перетину (interval scheduling),
код Гаффмана, алгоритми Kruskal/Prim/Dijkstra (Урок 5).

In [2]:
# Жадібно (правильно): максимум зустрічей без перетину.
# Секрет — сортувати за ЧАСОМ ЗАКІНЧЕННЯ і брати кожну сумісну.
def max_meetings(intervals):
    intervals = sorted(intervals, key=lambda x: x[1])   # за часом кінця
    chosen, end = [], float("-inf")
    for start, finish in intervals:
        if start >= end:                # не перетинається з останньою обраною
            chosen.append((start, finish))
            end = finish
    return chosen

meetings = [(1, 3), (2, 5), (4, 7), (1, 8), (5, 9), (8, 10)]
print("обрані зустрічі:", max_meetings(meetings))

# Жадібність, де вона ХИБНА: розмін монетами [1, 3, 4] на суму 6.
# Жадібно: 4 + 1 + 1 = 3 монети. Оптимально: 3 + 3 = 2 монети! (див. DP нижче)
print("\nдля монет [1,3,4] на 6 жадібність дає 3 монети, а треба 2 -> потрібне DP")

обрані зустрічі: [(1, 3), (4, 7), (8, 10)]

для монет [1,3,4] на 6 жадібність дає 3 монети, а треба 2 -> потрібне DP


## 3. 🔎 Динамічне програмування (DP)

**Динамічне програмування** застосовне, коли задача має:
1. **оптимальну підструктуру** — оптимум складається з оптимумів підзадач;
2. **підзадачі, що перекриваються** — ті самі підзадачі рахуються багато разів.

Ідея: **порахувати кожну підзадачу один раз** і **запам'ятати**. Два стилі:
- **Мемоізація (top-down)** — звичайна рекурсія + кеш (як `fib_memo` з Уроку 1);
- **Табуляція (bottom-up)** — заповнюємо таблицю від найменших підзадач до відповіді.

Розглянемо три класичні задачі: **розмін монет**, **рюкзак 0/1**, **найдовша спільна підпослідовність**.

In [3]:
# 3.1. Розмін монет: мінімум монет на суму (табуляція, bottom-up)
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount             # dp[s] = мін. монет на суму s
    for s in range(1, amount + 1):
        for c in coins:
            if c <= s and dp[s - c] + 1 < dp[s]:
                dp[s] = dp[s - c] + 1     # взяти монету c + оптимум для решти
    return dp[amount] if dp[amount] != INF else -1

print("монети [1,3,4], сума 6 -> мінімум монет:", coin_change([1, 3, 4], 6))   # 2 (3+3)
print("монети [2], сума 3 -> ", coin_change([2], 3))                            # -1 (неможливо)

монети [1,3,4], сума 6 -> мінімум монет: 2
монети [2], сума 3 ->  -1


In [4]:
# 3.2. Рюкзак 0/1: максимальна цінність за обмеженої ваги (кожен предмет 0 або 1 раз)
def knapsack(weights, values, capacity):
    n = len(weights)
    # dp[i][w] = макс. цінність з перших i предметів за ваги <= w
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i - 1][w]                    # не беремо предмет i
            if weights[i - 1] <= w:                    # або беремо, якщо влазить
                take = dp[i - 1][w - weights[i - 1]] + values[i - 1]
                dp[i][w] = max(dp[i][w], take)
    return dp[n][capacity]

w = [1, 3, 4, 5]
v = [1, 4, 5, 7]
print("рюкзак місткістю 7 -> макс. цінність:", knapsack(w, v, 7))   # 9 (предмети 3+4 -> ваги 3+4=7, цінність 4+5)

рюкзак місткістю 7 -> макс. цінність: 9


In [5]:
# 3.3. Найдовша спільна підпослідовність (LCS) двох рядків
def lcs(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1        # символи збіглись
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

print("LCS('ABCBDAB', 'BDCAB') =", lcs("ABCBDAB", "BDCAB"))   # 4 (напр. BCAB)
print("LCS('приклад', 'клапан') =", lcs("приклад", "клапан"))

LCS('ABCBDAB', 'BDCAB') = 4
LCS('приклад', 'клапан') = 3


## 4. Backtracking (перебір з відсіканням)

**Backtracking** — систематичний перебір усіх варіантів: будуємо розв'язок покроково, а щойно
крок веде в глухий кут — **відкочуємось** (backtrack) і пробуємо інший. Так генерують перестановки,
підмножини, розв'язують судоку, N ферзів, лабіринти.

Це, по суті, **DFS по дереву варіантів** з відсіканням гілок, що не можуть дати розв'язку.

In [6]:
# 4.1. Усі підмножини (power set) — на кожному кроці: взяти / не взяти елемент
def subsets(nums):
    result = []
    def backtrack(start, current):
        result.append(current[:])           # поточна підмножина
        for i in range(start, len(nums)):
            current.append(nums[i])         # взяти nums[i]
            backtrack(i + 1, current)
            current.pop()                   # відкотити (не взяти) -> пробуємо далі
    backtrack(0, [])
    return result

print("підмножини [1,2,3]:", subsets([1, 2, 3]))

# 4.2. Усі перестановки
def permutations(nums):
    result = []
    def backtrack(current, remaining):
        if not remaining:
            result.append(current[:]); return
        for i in range(len(remaining)):
            current.append(remaining[i])
            backtrack(current, remaining[:i] + remaining[i+1:])
            current.pop()
    backtrack([], nums)
    return result

print("перестановки [1,2,3]:", permutations([1, 2, 3]))

підмножини [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
перестановки [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


In [7]:
# 4.3. Задача N ферзів: скількома способами розставити N ферзів, щоб не били одне одного
def count_n_queens(n):
    count = 0
    cols, diag1, diag2 = set(), set(), set()   # зайняті стовпці й дві діагоналі
    def place(row):
        nonlocal count
        if row == n:
            count += 1; return                 # усі ряди заповнено -> розв'язок
        for col in range(n):
            if col in cols or (row - col) in diag1 or (row + col) in diag2:
                continue                        # б'ється -> відсікаємо
            cols.add(col); diag1.add(row - col); diag2.add(row + col)
            place(row + 1)                      # ставимо наступний ряд
            cols.discard(col); diag1.discard(row - col); diag2.discard(row + col)  # backtrack
    place(0)
    return count

for n in range(4, 9):
    print(f"N={n}: {count_n_queens(n)} розв'язків")   # 4:2, 5:10, 6:4, 7:40, 8:92

N=4: 2 розв'язків
N=5: 10 розв'язків
N=6: 4 розв'язків
N=7: 40 розв'язків
N=8: 92 розв'язків


## 5. 🚀 Middle+: короткий огляд «на десерт»

### Рядкові алгоритми
- **Наївний пошук підрядка** — O(n·m). **KMP** та **Rabin-Karp** роблять це за O(n+m) (KMP —
  через префікс-функцію; Rabin-Karp — через хешування «вікна»).
- **Trie** (Урок 3) — для багатьох підрядків/префіксів.

### Теорія чисел
- **НСД за Евклідом** — `gcd(a, b) = gcd(b, a % b)`;
- **Решето Ератосфена** — усі прості до `n` за O(n log log n).

### Робота з бітами (bit manipulation)
`&` (і), `|` (або), `^` (xor), `~` (не), `<<`/`>>` (зсуви). Трюки: `x & 1` — парність,
`x & (x-1)` прибирає останній одиничний біт, `x ^ x == 0`.

In [8]:
# НСД за Евклідом
def gcd(a, b):
    while b:
        a, b = b, a % b        # (a,b) -> (b, залишок)
    return a
print("gcd(48, 36) =", gcd(48, 36))     # 12

# Решето Ератосфена: усі прості до n
def sieve(n):
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(n ** 0.5) + 1):
        if is_prime[p]:
            for multiple in range(p * p, n + 1, p):
                is_prime[multiple] = False    # викреслюємо кратні
    return [i for i in range(n + 1) if is_prime[i]]
print("прості до 30:", sieve(30))

# Біти
print("13 у двійковому:", bin(13))            # 0b1101
print("13 парне? (13 & 1 == 0):", (13 & 1) == 0)
print("5 ^ 3 =", 5 ^ 3, "| 5 & 3 =", 5 & 3, "| 5 | 3 =", 5 | 3)

gcd(48, 36) = 12
прості до 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
13 у двійковому: 0b1101
13 парне? (13 & 1 == 0): False
5 ^ 3 = 6 | 5 & 3 = 1 | 5 | 3 = 7


# ✅ Підсумок уроку
- **Розділяй і володарюй** — поділити / розв'язати рекурсивно / об'єднати (fast power, merge/quick sort).
- **Жадібні** — локально найкращий вибір; швидко, але коректно лише для спеціальних задач (не для розміну [1,3,4]).
- **Динамічне програмування** — оптимальна підструктура + перекриття підзадач; мемоізація (top-down) або табуляція (bottom-up): розмін монет, рюкзак 0/1, LCS.
- **Backtracking** — DFS по дереву варіантів із відсіканням: підмножини, перестановки, N ферзів.
- **Десерт (Middle+):** KMP/Rabin-Karp (рядки), Евклід/решето (числа), бітові трюки.

### 🎉 Ви пройшли весь розділ «Алгоритми та структури даних»!
Далі — **практика**: [домашнє завдання](domashnie-zavdannia-algorytmy.ipynb) і задачники нижче.

### 📚 Хочу знати більше
- NeetCode (150 задач за патернами): <https://neetcode.io/>
- LeetCode: <https://leetcode.com/>
- CP-Algorithms: <https://cp-algorithms.com/>
- «Grokking Algorithms» (книга для початківців) та CLRS (академічний стандарт).